# Problema X00Y20Bx5T0 — Corpo Semi-Infinito 2D Aquecido em Metade da Superfície

[![Abrir no Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ana-mat-br/codigos-livro-conducao-calor/blob/main/x00y20bx5t0.ipynb)

**Livro:** *Condução de Calor: Aplicações das Funções de Green em Problemas de Engenharia*
**Autores:** Ana Paula Fernandes e Gilmar Guimarães

---

## Descrição do problema

Corpo semi-infinito bidimensional ($-\infty < x < \infty$, $y \ge 0$) com temperatura inicial nula, aquecido por um fluxo de calor constante $q''_0$ na **metade** da superfície ($x < 0$, $y = 0$) e isolado na outra metade. Na nomenclatura de Beck, a variação em degrau do fluxo ao longo da superfície é do tipo $Bx5$, e o problema é o **X00Y20Bx5T0**.

A função de Green é o produto $G_{X00} \cdot G_{Y20}$ — núcleo gaussiano do corpo infinito na direção $x$ e núcleo do meio semi-infinito com superfície do tipo 2 na direção $y$ (Cap. 3) — e a solução é

$$T(x,y,t) = \frac{\alpha q''_0}{k}\int_0^t\!\int_{-\infty}^{0} G_{X00Y20}(x,y,t\,|\,x',0,\tau)\,dx'\,d\tau$$

## Desenvolvimento das integrais

A integral do núcleo gaussiano sobre a metade aquecida dá a função erro complementar:

$$\int_{-\infty}^{0} G_{X00}(x,t\,|\,x',\tau)\,dx' = \frac{1}{2}\operatorname{erfc}\!\left(\frac{x}{\sqrt{4\alpha(t-\tau)}}\right)$$

Com a mudança de variável $t - \tau = t\,v^2$, a solução reduz-se a **uma única quadratura adimensional**:

$$T(x,y,t) = \frac{q''_0}{k}\sqrt{\frac{\alpha t}{\pi}}\int_0^1 \operatorname{erfc}\!\left(\frac{u_x}{v}\right)\exp\!\left(-\frac{u_y^2}{v^2}\right)dv, \qquad u_x = \frac{x}{\sqrt{4\alpha t}},\quad u_y = \frac{y}{\sqrt{4\alpha t}}$$

Na superfície ($y=0$) a integral tem **forma fechada** em termos da função exponencial integral $E_1(z) = \int_z^\infty (e^{-s}/s)\,ds$:

$$T(x,0,t) = \frac{q''_0}{k}\sqrt{\frac{\alpha t}{\pi}}\left[\operatorname{erfc}(u_x) - \frac{u_x}{\sqrt{\pi}}\,E_1(u_x^2)\right]$$

A dependência espacial dá-se apenas pela variável de similaridade $u_x$: perfis normalizados em instantes distintos colapsam numa **curva universal**.

## Bibliotecas utilizadas

- **NumPy**: computação numérica.
- **Matplotlib**: gráficos.
- **SciPy**: `erfc` (função erro complementar), `exp1` (exponencial integral $E_1$) e `quad` (quadratura adaptativa da integral em $v$).

In [ ]:
import numpy as np                        # computação numérica
import matplotlib.pyplot as plt           # gráficos
from scipy.special import erfc, exp1      # erfc e exponencial integral E1
from scipy.integrate import quad          # quadratura adaptativa

## Parâmetros e funções da solução

In [ ]:
alpha = 1.93e-7      # difusividade térmica [m²/s]
k     = 0.81         # condutividade térmica [W/(m·K)]
q0    = 1e4          # fluxo imposto na metade x<0 da superfície [W/m²]


def T_quadratura(x, y, t):
    """T(x,y,t) pela quadratura adimensional (válida em todo o domínio)."""
    ux = x/np.sqrt(4*alpha*t)
    uy = y/np.sqrt(4*alpha*t)
    integrando = lambda v: erfc(ux/v)*np.exp(-(uy/v)**2)
    val, _ = quad(integrando, 0.0, 1.0, limit=200)
    return (q0/k)*np.sqrt(alpha*t/np.pi)*val


def T_superficie(x, t):
    """T(x,0,t) pela forma fechada com a exponencial integral E1."""
    ux = x/np.sqrt(4*alpha*t)
    J = 1.0 if ux == 0.0 else erfc(ux) - (ux/np.sqrt(np.pi))*exp1(ux**2)
    return (q0/k)*np.sqrt(alpha*t/np.pi)*J


def T_1d_superficie(t):
    """Superfície do X20B1T0 (Cap. 3): superfície inteiramente aquecida."""
    return (q0/k)*np.sqrt(4*alpha*t)/np.sqrt(np.pi)

## Verificações intrínsecas

Três checagens, todas extraídas da própria solução:

1. **Interior da região aquecida** ($u_x \to -\infty$): recupera a temperatura superficial do problema 1D X20B1T0 (Cap. 3) — longe da borda, o corpo não "sente" a metade isolada;
2. **Borda da região aquecida** ($x=0$): exatamente **metade** do valor 1D;
3. **Superposição**: $T(x,0,t) + T(-x,0,t) = T_{1D}(t)$ — as duas metades complementares somam a superfície toda aquecida.

E, adicionalmente, a quadratura numérica deve coincidir com a forma fechada em $y=0$.

In [ ]:
t_chk = 50.0
d = np.sqrt(4*alpha*t_chk)
T1d = T_1d_superficie(t_chk)
print(f'1D (X20B1T0):            {T1d:10.4f} °C')
print(f'u_x -> -inf:             {T_superficie(-8*d, t_chk):10.4f} °C  (= 1D)')
print(f'borda x = 0:             {T_superficie(0.0, t_chk):10.4f} °C  (= 1D/2 = {T1d/2:.4f})')
soma = T_superficie(1.5*d, t_chk) + T_superficie(-1.5*d, t_chk)
print(f'T(x) + T(-x):            {soma:10.4f} °C  (= 1D)')
dif = max(abs(T_quadratura(u*d, 0.0, t_chk) - T_superficie(u*d, t_chk))
          for u in np.linspace(-3, 3, 25))
print(f'quadratura vs fechada:   {dif:10.2e} °C  (desvio máximo)')

## Curva universal da temperatura superficial

Normalizando pelo valor 1D, os perfis de todos os instantes colapsam na curva $[\operatorname{erfc}(u_x) - (u_x/\sqrt{\pi})E_1(u_x^2)]/2$.

In [ ]:
cores  = ['#D55E00', '#0072B2', '#009E73', '#CC79A7', '#E69F00']
marcas = ['o', 's', '^', 'd']

ux_curva = np.linspace(-3, 3, 400)
J = np.array([1.0 if u == 0 else erfc(u) - (u/np.sqrt(np.pi))*exp1(u**2)
              for u in ux_curva])

fig, ax = plt.subplots()
ax.plot(ux_curva, J/2, 'k-', linewidth=1.5, label='Forma fechada')
for tt, cor, mk in zip([5.0, 20.0, 50.0, 100.0], cores, marcas):
    dd = np.sqrt(4*alpha*tt)
    uu = np.linspace(-2.9, 2.9, 20)
    Tn = [T_quadratura(u*dd, 0.0, tt)/T_1d_superficie(tt) for u in uu]
    ax.plot(uu, Tn, mk, color=cor, markerfacecolor='none', markersize=5,
            label=f'Quadratura, t = {tt:.0f} s')
ax.axhline(0.5, color='gray', linestyle=':', linewidth=0.8)
ax.axvline(0.0, color='gray', linestyle=':', linewidth=0.8)
ax.set_xlabel('u_x = x/√(4αt)')
ax.set_ylabel('T(x, 0, t) / T_1D(t)')
ax.set_title('X00Y20Bx5T0 — curva universal da temperatura superficial')
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## Campo de temperatura no interior

A quadratura vale em todo o domínio. As isotermas em $t = 50$ s mostram o calor entrando pela metade aquecida ($x<0$) e difundindo-se lateralmente por baixo da metade isolada ($x>0$).

In [ ]:
tt = 50.0
dd = np.sqrt(4*alpha*tt)
xs = np.linspace(-3*dd, 3*dd, 81)
ys = np.linspace(0, 2.5*dd, 41)
TT = np.array([[T_quadratura(x, y, tt) for x in xs] for y in ys])

fig, ax = plt.subplots(figsize=(9, 4))
cf = ax.contourf(xs*1e3, ys*1e3, TT, levels=15, cmap='inferno')
cs = ax.contour(xs*1e3, ys*1e3, TT, levels=8, colors='white',
                linewidths=0.6)
ax.clabel(cs, fontsize=7, fmt='%.0f')
fig.colorbar(cf, ax=ax, label='T (°C)')
ax.invert_yaxis()                      # y cresce para dentro do corpo
ax.set_xlabel('x (mm)')
ax.set_ylabel('y (mm)')
ax.set_title(f'Campo de temperatura em t = {tt:.0f} s '
             '(fluxo em x < 0, isolado em x > 0)')
plt.tight_layout()
plt.show()

---

## Conclusão

Partindo da solução geral por funções de Green da Seção 5.4, o desenvolvimento das integrais produziu: uma quadratura adimensional única válida em todo o domínio, uma forma fechada na superfície com a exponencial integral $E_1$ e uma curva universal de similaridade. As três verificações intrínsecas — limite 1D, meio valor na borda e superposição das metades complementares — confirmam a solução dentro do próprio método das funções de Green, e a quadratura coincide com a forma fechada em precisão de máquina.